# 🪶 Automated Feather Measurement and Analysis Pipeline
This notebook runs a **minimal slice** of the distributed pipeline from a Jupyter session while keeping dataset storage and heavy inference on the Apple Silicon cluster. The notebook acts as the control plane: select one source image, dispatch processing, and review returned feather crops.


## Step 1: Laboratory Equipment Setup (Environment & Cluster Configuration)
This step configures cluster endpoints, sets Celery broker/backend variables, initializes a lightweight task client, and prepares a local preview directory. Use environment variables to override hostnames, SSH keys, and remote data paths without editing notebook code.


In [ ]:
import os
import shlex
import socket
import subprocess
import time
from pathlib import Path
from urllib.parse import urlparse

from celery import Celery
from IPython.display import display
from PIL import Image

# Cluster + path config (override with env vars as needed).
HEAD_HOST = os.getenv('FEATHER_HEAD_HOST', '10.0.0.148')
SSH_USER = os.getenv('FEATHER_SSH_USER', 'openteams')
KEY_PATH = os.path.expanduser(os.getenv('FEATHER_SSH_KEY', '~/.ssh/ubuntu-mac-openteams-admin'))
REMOTE_REPO_DIR = os.getenv('FEATHER_REMOTE_REPO_DIR', '/Users/openteams/Feather_Molt_Project')
REMOTE_INPUT_DIR = os.getenv('FEATHER_REMOTE_INPUT_DIR', f'{REMOTE_REPO_DIR}/data/raw')
REMOTE_OUTPUT_DIR = os.getenv('FEATHER_REMOTE_OUTPUT_DIR', f'{REMOTE_REPO_DIR}/data/processed')
CLUSTER_HOSTS = [h.strip() for h in os.getenv('FEATHER_CLUSTER_HOSTS', '10.0.0.148,10.0.0.63,10.0.0.19,10.0.0.118').split(',') if h.strip()]

# Celery broker/backend reachable from this notebook host.
BROKER_URL = os.getenv('BROKER_URL', f'redis://{HEAD_HOST}:6379/0')
RESULT_BACKEND = os.getenv('RESULT_BACKEND', '')
if RESULT_BACKEND:
    os.environ['RESULT_BACKEND'] = RESULT_BACKEND
os.environ['BROKER_URL'] = BROKER_URL

def assert_tcp_reachable(url: str, name: str, timeout: float = 2.0):
    parsed = urlparse(url)
    host = parsed.hostname
    port = parsed.port or 6379
    if not host:
        raise RuntimeError(f'{name} URL is invalid: {url}')
    s = socket.socket()
    s.settimeout(timeout)
    try:
        s.connect((host, port))
    except Exception as exc:
        raise RuntimeError(
            f'{name} not reachable at {host}:{port} from this notebook host. '
            f'Original error: {type(exc).__name__}: {exc}'
        ) from exc
    finally:
        s.close()

assert_tcp_reachable(BROKER_URL, 'BROKER_URL')

# Use result backend only if explicitly set and reachable.
backend = None
if RESULT_BACKEND:
    assert_tcp_reachable(RESULT_BACKEND, 'RESULT_BACKEND')
    backend = RESULT_BACKEND

celery_app = Celery('feather_pipeline', broker=BROKER_URL, backend=backend)

LOCAL_PREVIEW_DIR = Path('./.minimal_slice_preview').resolve()
LOCAL_PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

print('HEAD_HOST =', HEAD_HOST)
print('CLUSTER_HOSTS =', CLUSTER_HOSTS)
print('BROKER_URL =', BROKER_URL)
print('RESULT_BACKEND =', RESULT_BACKEND or '(disabled)')
print('REMOTE_INPUT_DIR =', REMOTE_INPUT_DIR)
print('REMOTE_OUTPUT_DIR =', REMOTE_OUTPUT_DIR)
print('LOCAL_PREVIEW_DIR =', LOCAL_PREVIEW_DIR)


## Step 2: Remote Inventory and Transport Helpers
Helper functions provide cluster-safe file discovery and transfer operations over SSH/SCP. This enables image discovery and result collection directly from cluster nodes rather than mirroring the full dataset locally.


In [ ]:
def _ssh_base_cmd(host: str):
    cmd = ['ssh']
    if KEY_PATH and os.path.exists(KEY_PATH):
        cmd += ['-i', KEY_PATH]
    cmd += ['-o', 'StrictHostKeyChecking=no', f'{SSH_USER}@{host}']
    return cmd

def ssh_lines(host: str, remote_cmd: str):
    out = subprocess.check_output([*_ssh_base_cmd(host), remote_cmd], text=True)
    return [line.strip() for line in out.splitlines() if line.strip()]

def list_remote_images(input_host: str = HEAD_HOST):
    d = shlex.quote(REMOTE_INPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f \\( -iname '*.jpg' -o -iname '*.jpeg' \\) | sort"
    return ssh_lines(input_host, cmd)

def list_remote_outputs(host: str):
    d = shlex.quote(REMOTE_OUTPUT_DIR)
    cmd = f"find {d} -maxdepth 1 -type f -iname '*.jpg' | sort"
    return ssh_lines(host, cmd)

def fetch_remote_file(host: str, remote_path: str, local_dir: Path):
    local_dir.mkdir(parents=True, exist_ok=True)
    local_file = local_dir / Path(remote_path).name
    remote_cmd = f"cat {shlex.quote(remote_path)}"
    with open(local_file, 'wb') as f:
        subprocess.check_call([*_ssh_base_cmd(host), remote_cmd], stdout=f)


## Step 3: Minimal Slice Dispatch (Single-Image Distributed Processing)
A single source image is selected from the remote input directory and submitted as a Celery task. Workers perform Grounding DINO + SAM extraction and metadata handling on the cluster, then return task status to this notebook.

**Integration Note:** this is the insertion point for broader batch runs or alternate orchestration, while preserving the same remote execution model.


In [ ]:
remote_images = list_remote_images(HEAD_HOST)
print(f'Found {len(remote_images)} remote images on {HEAD_HOST}.')
if not remote_images:
    raise RuntimeError('No remote images found. Check FEATHER_REMOTE_INPUT_DIR.')

SAMPLE_INDEX = 0  # Change this if you want a different source image.
image_path = remote_images[SAMPLE_INDEX]
source_stem = Path(image_path).stem
print('Selected remote source:', image_path)

# Fire-and-forget dispatch; completion is detected by output polling in Step 4.
async_result = celery_app.send_task(
    'src.celery_tasks.process_image',
    args=[image_path, REMOTE_OUTPUT_DIR],
    ignore_result=True,
)
print('Task dispatched. ID:', async_result.id)


## Step 4: Retrieval and Visual Review
After task completion, this step searches cluster nodes for generated feather crops, pulls matching outputs to a local preview directory, and renders them inline for QA.


In [ ]:
prefix = f"{source_stem}_"
matching = []

WAIT_TIMEOUT_SECONDS = int(os.getenv('FEATHER_WAIT_TIMEOUT_SECONDS', '180'))
POLL_SECONDS = float(os.getenv('FEATHER_POLL_SECONDS', '3'))
deadline = time.time() + WAIT_TIMEOUT_SECONDS

while time.time() < deadline and not matching:
    for host in CLUSTER_HOSTS:
        try:
            out_files = list_remote_outputs(host)
        except Exception as exc:
            print(f'Skipping host {host}: {exc}')
            continue

        host_matches = []
        for p in out_files:
            name = Path(p).name
            if name.startswith(prefix) and ('_Feather_' in name or '_BoundingBoxes' in name):
                host_matches.append(p)

        if host_matches:
            print(f'Found {len(host_matches)} segment(s) on {host}.')
            matching = [(host, p) for p in host_matches]
            break

    if not matching:
        print('Waiting for output files...')
        time.sleep(POLL_SECONDS)

if not matching:
    raise RuntimeError(
        f'No matching segment outputs found within {WAIT_TIMEOUT_SECONDS}s. '
        'Check worker status and remote output path.'
    )

preview_dir = LOCAL_PREVIEW_DIR / source_stem
preview_dir.mkdir(parents=True, exist_ok=True)

for host, remote_file in matching:
    fetch_remote_file(host, remote_file, preview_dir)

local_files = sorted(preview_dir.glob('*.jpg'))
print(f'Fetched {len(local_files)} files into {preview_dir}')

for f in local_files:
    print(f.name)
    display(Image.open(f))


## Step 5: Color Pattern Analysis (Native R "pavo" Subprocess)
The extracted feather is routed to standard biological analysis software (`pavo`). This dynamically reads the segmented feather, classifies the color patches into distinct groups, and returns the resulting color and pattern statistics.

**Understanding the Output:**
- **The Images:** The output displays the original feather next to a "classified" version. The classified version reduces the thousands of natural colors down to the 3 dominant biological pigment groups (e.g., dark melanin, light melanin, and background).
- **The Statistics (`adj_stats`):** The printed numbers represent "Adjacency Statistics". This is a mathematical way of measuring the pattern of the feather. It calculates how frequently a dark patch touches a light patch versus how often a dark patch touches another dark patch. This provides a quantifiable metric to describe complex patterns like barring, spotting, or mottling!


In [ ]:
import subprocess
import os
import sys
import matplotlib.pyplot as plt
import cv2
import numpy as np

# Select the first feather from the fetched files to analyze
feather_files = [f for f in local_files if "_Feather_" in f.name]
if not feather_files:
    raise RuntimeError("No feather crops found to analyze.")
crop_path = str(feather_files[0])

r_script = f"""
suppressMessages(library(pavo))

cat('Analyzing segmented crop directly in R: {crop_path}\n')

# 1. LOAD IMAGE
feather_img <- getimg('{crop_path}')

# 2. CLASSIFY BIOLOGICAL PATCHES
classified_feather <- classify(feather_img, kcols = 3)

# 3. EXPORT RAW MATRIX DATA FOR PYTHON
# Bypassing R's graphical plotting to prevent dimension stretching!
write.table(attr(classified_feather, "classRGB"), "class_colors.csv", row.names=FALSE, col.names=FALSE, sep=",")
write.table(as.matrix(classified_feather), "class_matrix.csv", row.names=FALSE, col.names=FALSE, sep=",")

# 4. OUTPUT ADJACENCY STATISTICS
adj_stats <- adjacent(classified_feather, xscale = 1)
print(adj_stats)
"""

env = os.environ.copy()
if sys.platform == "darwin":
    env['R_HOME'] = "/Library/Frameworks/R.framework/Resources"
    env['PATH'] = env['R_HOME'] + "/bin:" + env.get('PATH', "")

print("Executing pavo R script...")
try:
    result = subprocess.run(['Rscript', '-e', r_script], capture_output=True, text=True, env=env)
    print(result.stdout)
    if result.stderr:
        print("R stderr:", result.stderr)
except FileNotFoundError:
    print("Rscript not found on this system. Skipping pavo analysis.")

# ---------------------------------------------------------
# RENDER WITH MATPLOTLIB FOR PERFECT DISPLAY
# ---------------------------------------------------------
if os.path.exists('class_matrix.csv') and os.path.exists('class_colors.csv'):
    # Load original un-stretched image directly from Python
    img1 = cv2.imread(crop_path)[:, :, ::-1] # BGR to RGB
    
    # Reconstruct the classified image manually
    matrix = np.loadtxt('class_matrix.csv', delimiter=',')
    colors = np.loadtxt('class_colors.csv', delimiter=',') # R, G, B floats
    
    # Create empty image
    img2 = np.zeros((matrix.shape[0], matrix.shape[1], 3))
    
    # Map classes to colors (Classes are 1, 2, 3 in R)
    for i in range(1, len(colors) + 1):
        img2[matrix == i] = colors[i-1]
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 10))
    
    ax1.imshow(img1)
    ax1.set_title("1. Segmented Crop", fontsize=14, pad=15)
    ax1.axis('off')
    
    ax2.imshow(img2)
    ax2.set_title("2. pavo k=3 Color Classification", fontsize=14, pad=15)
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Clean up temp files
    os.remove('class_matrix.csv')
    os.remove('class_colors.csv')
